In [3]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')
test_pred = pd.read_csv(PROJECT_ROOT.parent / 'data' / '0_final_result.csv')
# test_infer_pred = pd.read_csv(PROJECT_ROOT.parent / 'data' / 'final_result_infer.csv')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드



In [4]:
# ============================================================
# 1-3. 최종 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 독립변수군
feature_cols = [
    col for col in test.columns
    if col not in ('timestamp', 'event_id', target_col)
]

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

In [5]:
# test_pred와 복원한 y_test 매칭하기

print(test.index)
print(test_pred.index)

test['test_pred'] = (
    test_pred
    .set_index('row_id')['pred_maxrise_6h']
    .reindex(test.index)
)
print(test.shape, test_pred.shape)


from sklearn.metrics import root_mean_squared_error

test_rmse = root_mean_squared_error(
    test[target_col], test['test_pred']
)

print('test_rmse: ', test_rmse)

Index(['TE_000046543', 'TE_000111013', 'TE_000086280', 'TE_000136298',
       'TE_000166154', 'TE_000114804', 'TE_000082963', 'TE_000122824',
       'TE_000083061', 'TE_000125812',
       ...
       'TE_000026376', 'TE_000111372', 'TE_000019025', 'TE_000111821',
       'TE_000114211', 'TE_000014947', 'TE_000025081', 'TE_000001900',
       'TE_000095649', 'TE_000092086'],
      dtype='str', name='row_id', length=114111)
RangeIndex(start=0, stop=186263, step=1)
(114111, 35) (186263, 2)
test_rmse:  0.20231018043229732


In [3]:
# test_pred와 복원한 y_test_infer 매칭하기

test['test_pred'] = (
    test_infer_pred
    .set_index('row_id')['pred_maxrise_6h']
    .reindex(test.index)
)

from sklearn.metrics import root_mean_squared_error

test_rmse = root_mean_squared_error(
    test[target_col], test['test_pred']
)

print('test_rmse: ', test_rmse)

test_rmse:  0.20231018043229732
